In [1]:
import os
import glob
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split

# Set random seed for scientific reproducibility
np.random.seed(42)

def get_image_files(directory):
    """Helper to list all images in a directory efficiently."""
    # Strip any potential accidental leading/trailing whitespace from paths
    directory = directory.strip()
    if not os.path.exists(directory):
        print(f"⚠️ Warning: Directory does not exist -> '{directory}'")
        return []
    extensions = ('*.jpg', '*.jpeg', '*.png', '*.bmp', '*.tif', '*.tiff')
    files = []
    for ext in extensions:
        files.extend(glob.glob(os.path.join(directory, ext)))
        files.extend(glob.glob(os.path.join(directory, ext.upper())))
    return sorted(files)

# Master storage registry
metadata_records = []

print("=== STARTING DATASET SCAN AND SUBSAMPLING ===")

# =====================================================================
# 1. DF2023 Train Subset (Strict Limit: 10,000 items)
# =====================================================================
df2023_train_img_dir = "/kaggle/input/datasets/hiwotyirga/the-digital-forensics-2023-dataset-df2023/DF2023_V15_train/COCO_V15"
df2023_train_gt_dir = "/kaggle/input/datasets/hiwotyirga/the-digital-forensics-2023-dataset-df2023/DF2023_V15_train/COCO_V15_GT"

df2023_train_imgs = get_image_files(df2023_train_img_dir)
print(f"Found {len(df2023_train_imgs)} files in DF2023 Train")

if df2023_train_imgs:
    train_sample_size = min(10000, len(df2023_train_imgs))
    sampled_train_imgs = np.random.choice(df2023_train_imgs, size=train_sample_size, replace=False)
    
    for img_path in sampled_train_imgs:
        base_name = os.path.basename(img_path)
        gt_path = os.path.join(df2023_train_gt_dir, base_name)
        metadata_records.append({
            'image_path': img_path,
            'mask_path': gt_path if os.path.exists(gt_path) else None,
            'source_dataset': 'DF2023_Train',
            'category': 'manipulation',
            'split': 'train'
        })

# =====================================================================
# 2. DF2023 Validation & Internal Test Subset (Strict Limit: 3,000 items)
# =====================================================================
df2023_val_img_dir = "/kaggle/input/datasets/mubarekahmed/df2023-subset/DF2023_V15_val/COCO_V15"
df2023_val_gt_dir = "/kaggle/input/datasets/mubarekahmed/df2023-subset/DF2023_V15_val/COCO_V15_GT"

df2023_val_imgs = get_image_files(df2023_val_img_dir)
print(f"Found {len(df2023_val_imgs)} files in DF2023 Val")

if df2023_val_imgs:
    val_sample_size = min(3000, len(df2023_val_imgs))
    sampled_val_imgs = np.random.choice(df2023_val_imgs, size=val_sample_size, replace=False)
    
    df2023_v_idx, df2023_t_idx = train_test_split(sampled_val_imgs, test_size=0.50, random_state=42)
    
    for img_list, split_label in [(df2023_v_idx, 'val'), (df2023_t_idx, 'test')]:
        for img_path in img_list:
            base_name = os.path.basename(img_path)
            gt_path = os.path.join(df2023_val_gt_dir, base_name) 
            metadata_records.append({
                'image_path': img_path,
                'mask_path': gt_path if os.path.exists(gt_path) else None,
                'source_dataset': 'DF2023_Val',
                'category': 'manipulation',
                'split': split_label
            })

# =====================================================================
# 3. SIDA Forensics Subset (Limit to 6,000 items total -> 70/15/15 Split)
# =====================================================================
sida_paths = {
    'full_synthetic': "/kaggle/input/datasets/mubarekahmed/sida-forensics/full_synthetic",
    'masks': "/kaggle/input/datasets/mubarekahmed/sida-forensics/masks",
    'real': "/kaggle/input/datasets/mubarekahmed/sida-forensics/real",
    'tampered': "/kaggle/input/datasets/mubarekahmed/sida-forensics/tampered"
}

sida_all_records = []
for category, path in sida_paths.items():
    sida_imgs = get_image_files(path)
    print(f"Found {len(sida_imgs)} files in SIDA category: {category}")
    for img_path in sida_imgs:
        sida_all_records.append({
            'image_path': img_path,
            'mask_path': None,
            'source_dataset': 'sida-forensics',
            'category': category
        })

sida_df_pool = pd.DataFrame(sida_all_records)
if not sida_df_pool.empty:
    _, sampled_sida_df = train_test_split(
        sida_df_pool, 
        test_size=min(6000, len(sida_df_pool)), 
        stratify=sida_df_pool['category'], 
        random_state=42
    )
    
    sida_train, sida_temp = train_test_split(sampled_sida_df, test_size=0.30, stratify=sampled_sida_df['category'], random_state=42)
    sida_val, sida_test = train_test_split(sida_temp, test_size=0.50, stratify=sida_temp['category'], random_state=42)
    
    sida_train['split'] = 'train'
    sida_val['split'] = 'val'
    sida_test['split'] = 'test'
    
    metadata_records.extend(pd.concat([sida_train, sida_val, sida_test]).to_dict(orient='records'))

# =====================================================================
# 4. So-Fake Setted Combined Subset (Read Parquet Files -> 2k items per split)
# =====================================================================
so_fake_paths = {
    'train': "/kaggle/input/datasets/mubarekahmed/so-fake-setted/so_fake_combined/train",
    'validation': "/kaggle/input/datasets/mubarekahmed/so-fake-setted/so_fake_combined/validation",
    'ood_test': "/kaggle/input/datasets/mubarekahmed/so-fake-setted/so_fake_combined/ood_test"
}

for split_name, path in so_fake_paths.items():
    path = path.strip()
    parquet_files = sorted(glob.glob(os.path.join(path, "*.parquet")))
    print(f"Found {len(parquet_files)} parquet shards in So-Fake: {split_name}")
    
    if parquet_files:
        split_records = []
        for p_file in parquet_files:
            if len(split_records) >= 2000:
                break
            
            df_parquet = pd.read_parquet(p_file)
            
            for _, row in df_parquet.iterrows():
                if len(split_records) >= 2000:
                    break
                
                img_p = row.get('image_path', row.get('path', None))
                mask_p = row.get('mask_path', row.get('mask', None))
                cat = row.get('category', 'synthetic_hybrid')
                
                split_records.append({
                    'image_path': img_p if img_p else p_file,
                    'mask_path': mask_p if pd.notna(mask_p) else None,
                    'source_dataset': 'so-fake-combined',
                    'category': cat,
                    'split': 'val' if split_name == 'validation' else ('test' if split_name == 'ood_test' else 'train')
                })
        
        metadata_records.extend(split_records)

# =====================================================================
# Compile and Verify Final Architecture Safely
# =====================================================================
columns = ['image_path', 'mask_path', 'source_dataset', 'category', 'split']
final_metadata_df = pd.DataFrame(metadata_records, columns=columns)

# Save dataframe out to working directory
final_metadata_df.to_csv("balanced_dataset_metadata.csv", index=False)

print("\n=== FINAL GENERATED METADATA MATRIX ===")
if not final_metadata_df.empty:
    print(final_metadata_df.groupby(['source_dataset', 'split']).size())
else:
    print("❌ Critical: Dataset processing ended with 0 total indexed items.")

=== STARTING DATASET SCAN AND SUBSAMPLING ===
Found 1000000 files in DF2023 Train
Found 5000 files in DF2023 Val
Found 10000 files in SIDA category: full_synthetic
Found 10000 files in SIDA category: masks
Found 10000 files in SIDA category: real
Found 10000 files in SIDA category: tampered
Found 10 parquet shards in So-Fake: train
Found 4 parquet shards in So-Fake: validation
Found 4 parquet shards in So-Fake: ood_test

=== FINAL GENERATED METADATA MATRIX ===
source_dataset    split
DF2023_Train      train    10000
DF2023_Val        test      1500
                  val       1500
sida-forensics    test       900
                  train     4200
                  val        900
so-fake-combined  test      1912
                  train     2000
                  val       2000
dtype: int64
